# ✈️ Predicting Airline Satisfaction
**Kaggle Playground Series – Saison 6, Épisode 10** · Classification binaire · Métrique : **ROC AUC**

Objectif : prédire la probabilité qu'un passager soit satisfait de son vol à partir de ses notes de service, de son profil et de son trajet.

**Plan du notebook**
1. Configuration & imports
2. Fonctions utilitaires
3. Chargement des données
4. Exploration (EDA)
5. Nettoyage
6. Corrélations & information mutuelle
7. Encodage & score de référence
8. Expériences de feature engineering – round 1 (split simple)
9. Expériences – round 2 (split stratifié + target encoding)
10. Modèle final & fichier de soumission

## 1. Configuration & imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBClassifier

# --- Chemins (adapter DATA_DIR si besoin) ---
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = Path("submission.csv")

# --- Constantes ---
TARGET = "satisfaction"
ID_COLUMN = "id"
SEED_FEATURES = 0   # MI, KMeans, KFold du target encoding
SEED_MODEL = 1      # split train/validation et XGBoost

## 2. Fonctions utilitaires

### 2.1 Analyse : information mutuelle

In [ ]:
def compute_mi_scores(data):
    """Information mutuelle entre chaque variable et la cible (variables texte factorisées)."""
    X = data.copy()
    y = X.pop(TARGET)
    for col in X.select_dtypes("object"):
        X[col], _ = X[col].factorize()
    discrete_features = [pd.api.types.is_integer_dtype(dtype) for dtype in X.dtypes]
    scores = mutual_info_classif(X, y, discrete_features=discrete_features, random_state=SEED_FEATURES)
    scores = pd.Series(scores, name="Mi Score", index=X.columns)
    return scores.sort_values(ascending=False)


def plot_mi_scores(scores):
    scores = scores.sort_values(ascending=True)
    positions = np.arange(len(scores))
    plt.barh(positions, scores)
    plt.yticks(positions, list(scores.index))
    plt.title("Mutual Information Scores")

### 2.2 Préparation : encodage des variables catégorielles

In [ ]:
def encode_categoricals(data):
    """Transforme les colonnes texte/catégorielles en codes entiers (ordre alphabétique)."""
    data = data.drop(columns=[ID_COLUMN], errors="ignore")
    for col in data.select_dtypes(["object", "string", "category"]).columns:
        data[col] = data[col].astype("category").cat.codes
    return data

### 2.3 Évaluation : deux protocoles de validation
- **`score_simple_split`** (round 1) : split aléatoire 75/25, XGBoost par défaut.
- **`score_stratified_split`** (round 2) : split stratifié sur la cible, XGBoost `hist`, et possibilité de passer une fonction `feature_fn(train, valid)` pour créer des features **apprises uniquement sur le train**.

⚠️ Les deux protocoles ne sont pas strictement comparables entre eux (split différent).

In [ ]:
def score_simple_split(data):
    """ROC AUC sur un split aléatoire 75/25 (XGBoost par défaut)."""
    X = data.copy()
    y = X.pop(TARGET)
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=SEED_MODEL)
    model = XGBClassifier(random_state=SEED_MODEL)
    model.fit(X_train, y_train)
    valid_proba = model.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, valid_proba)


def score_stratified_split(data, feature_fn=None, target=TARGET):
    """ROC AUC sur un split stratifié 75/25.

    feature_fn : fonction optionnelle feature_fn(train, valid) -> (train, valid) qui crée
                 des features (apprises uniquement sur train).
    """
    train, valid = train_test_split(
        data, test_size=0.25, random_state=SEED_MODEL, stratify=data[target]
    )
    train, valid = train.copy(), valid.copy()
    if feature_fn is not None:
        train, valid = feature_fn(train, valid)

    y_train, y_valid = train.pop(target), valid.pop(target)
    model = XGBClassifier(random_state=SEED_MODEL, n_jobs=-1, tree_method="hist")
    model.fit(train, y_train)
    valid_proba = model.predict_proba(valid)[:, 1]
    return roc_auc_score(y_valid, valid_proba)

### 2.4 Feature engineering : clusters, PCA, target encoding

In [ ]:
def make_cluster_label(data, features, n_clusters, column_name):
    """KMeans sur `features` -> DataFrame (même index que `data`) avec le label de cluster."""
    kmeans = KMeans(n_clusters=n_clusters, n_init=80, random_state=SEED_FEATURES)
    labels = pd.DataFrame(index=data.index)
    labels[column_name] = kmeans.fit_predict(data[features])
    return labels


def make_centroid_distances(data, features, n_clusters):
    """KMeans sur `features` -> distance de chaque ligne à chacun des centroïdes."""
    kmeans = KMeans(n_clusters=n_clusters, n_init=80, random_state=SEED_FEATURES)
    distances = kmeans.fit_transform(data[features])
    return pd.DataFrame(distances, columns=[f"Centroid{i}" for i in range(distances.shape[1])])


def apply_pca(data, standardize=True):
    """PCA (optionnellement sur données standardisées) -> (modèle, composantes, loadings)."""
    data = data.copy()
    if standardize:
        data = (data - data.mean(axis=0)) / data.std(axis=0)
    pca = PCA()
    components = pca.fit_transform(data)
    component_names = [f"PCA{i + 1}" for i in range(components.shape[1])]
    components = pd.DataFrame(components, columns=component_names)
    loadings = pd.DataFrame(pca.components_.T, columns=component_names, index=data.columns)
    return pca, components, loadings


def make_pca_components(data, features):
    """Composantes principales calculées sur le sous-ensemble `features`."""
    _, components, _ = apply_pca(data.loc[:, features])
    return components


def make_target_encoding(col, new_name, target=TARGET, smoothing=20, n_splits=5):
    """Fabrique une fonction feature_fn(train, valid) qui ajoute un target encoding lissé.

    - train : encodage out-of-fold (KFold) pour éviter la fuite de la cible
    - valid/test : encodage calculé sur l'ensemble du train
    - catégories inconnues : remplacées par la moyenne globale de la cible (prior)
    """
    def feature_fn(train, valid):
        prior = train[target].mean()

        def encode_with(source, destination):
            stats = source.groupby(col)[target].agg(["sum", "count"])
            smoothed_mean = (stats["sum"] + prior * smoothing) / (stats["count"] + smoothing)
            return destination[col].map(smoothed_mean).fillna(prior)

        out_of_fold = pd.Series(index=train.index, dtype=float)
        folds = KFold(n_splits, shuffle=True, random_state=SEED_FEATURES)
        for fit_idx, apply_idx in folds.split(train):
            out_of_fold.iloc[apply_idx] = encode_with(train.iloc[fit_idx], train.iloc[apply_idx]).values
        train[new_name] = out_of_fold
        valid[new_name] = encode_with(train, valid)
        return train, valid
    return feature_fn

## 3. Chargement des données

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, index_col=ID_COLUMN)
print(train_df.shape)
train_df.head()

## 4. Exploration (EDA)

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.duplicated().sum()

In [ ]:
train_df.describe()

In [ ]:
plt.figure(figsize=(20, 10))

plt.subplot(2, 2, 1)
sns.histplot(train_df["Age"])
plt.title("Age Distribution")

plt.subplot(2, 2, 2)
sns.boxplot(train_df["Food and drink"])
plt.title("Food and drink Distribution")

plt.subplot(2, 2, 3)
sns.boxplot(train_df["On-board service"])
plt.title("On-board service Distribution")

plt.show()

In [ ]:
train_df["Gender"].value_counts(normalize=True) * 100

In [ ]:
train_df["Class"].value_counts(normalize=True) * 100

In [ ]:
sns.barplot(x=train_df[TARGET], y=train_df["Class"])

## 5. Nettoyage
Seule `Arrival Delay in Minutes` contient des valeurs manquantes (292 lignes sur 699 635) : elles sont supprimées.

In [ ]:
train_df = train_df.dropna()
train_df.isnull().sum()

In [ ]:
train_df.info()

In [ ]:
train_df["Arrival Delay in Minutes"].unique()

## 6. Corrélations & information mutuelle

In [ ]:
# Comme dans le code d'origine : la comparaison de dtype avec le tuple ne retient que les colonnes int64
# (la colonne float `Arrival Delay in Minutes` n'apparaît donc pas dans la heatmap).
numeric_cols = [col for col in train_df.columns if train_df[col].dtypes == ("int64", "float64")]
correlation_matrix = train_df[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".1f")

In [ ]:
# Calculée avant l'encodage (les colonnes texte sont factorisées dans la fonction)
mi_scores = compute_mi_scores(train_df)
plot_mi_scores(mi_scores)

In [ ]:
train_df.nunique()

## 7. Encodage & score de référence (baseline)
Les variables catégorielles (`Gender`, `Customer Type`, `Type of Travel`, `Class`, `satisfaction`) sont encodées en entiers, puis on mesure le score de base d'un XGBoost par défaut.

In [ ]:
train_encoded = encode_categoricals(train_df)
train_encoded.head()

In [ ]:
baseline_score = score_simple_split(train_encoded)
baseline_score

## 8. Feature engineering – Round 1 (split simple)
Chaque hypothèse est testée en ajoutant la/les nouvelle(s) feature(s) à `train_encoded` puis en comparant le ROC AUC au score de base (**0.95843**).

### 8.1 Cluster de « services » (KMeans, k = 4)
**Hypothèse** : regrouper les passagers par profil de notes de service (+ classe) donne un signal que les arbres ne trouvent pas seuls.  
**Résultat d'origine** : 0.95850

In [ ]:
service_features = [
    "On-board service",
    "Leg room service",
    "Inflight wifi service",
    "Checkin service",
    "Class",
]
service_cluster = make_cluster_label(
    train_encoded, service_features, n_clusters=4, column_name="Service_cluster"
)
X_service_cluster = train_encoded.join(service_cluster)
score_simple_split(X_service_cluster)

### 8.2 Distances aux centroïdes (KMeans, k = 6)
**Hypothèse** : les distances aux centres de clusters apportent plus d'information qu'un simple label.  
**Résultat d'origine** : 0.95776 (en dessous de la baseline)

> ⚠️ `centroid_distances` a un index `0..n-1` alors que `train_encoded` est indexé par `id` (avec des trous après le `dropna`) : la jointure n'est pas alignée ligne à ligne. Conservé tel quel (logique d'origine).

In [ ]:
centroid_distances = make_centroid_distances(train_encoded, service_features, n_clusters=6)
X_centroids = train_encoded.join(centroid_distances)
score_simple_split(X_centroids)

### 8.3 Moyenne de la cible par classe (`sat_class`)
**Hypothèse** : donner explicitement au modèle le taux de satisfaction moyen de chaque classe (Business / Eco / Eco Plus) l'aide.  
**Résultat d'origine** : 0.95839 (pas de gain)

In [ ]:
X_class_target_mean = train_encoded.copy()
X_class_target_mean["sat_class"] = X_class_target_mean.groupby("Class")[TARGET].transform("mean")
X_class_target_mean["sat_class"].unique()

In [ ]:
score_simple_split(X_class_target_mean)

### 8.4 Cluster de « retards » (KMeans, k = 2) + suppression des variables de retard
**Hypothèse** : les variables de retard sont faiblement informatives (cf. information mutuelle) ; les résumer en un cluster, voire les supprimer, ne devrait pas dégrader le modèle.  
**Résultats d'origine** : 0.95854 avec `Service_cluster` + `Delay_cluster` · 0.95852 après suppression des 3 variables de retard / horaire + `Delay_cluster`

In [ ]:
delay_features = [
    "Arrival Delay in Minutes",
    "Departure/Arrival time convenient",
    "Departure Delay in Minutes",
]
delay_cluster = make_cluster_label(
    train_encoded, delay_features, n_clusters=2, column_name="Delay_cluster"
)
X_both_clusters = train_encoded.join(service_cluster).join(delay_cluster)
score_simple_split(X_both_clusters)

In [ ]:
X_without_delay = X_both_clusters.drop(columns=[
    "Delay_cluster",
    "Arrival Delay in Minutes",
    "Departure/Arrival time convenient",
    "Departure Delay in Minutes",
])
score_simple_split(X_without_delay)

### 8.5 PCA sur les variables de retard
**Hypothèse** : résumer les variables de retard corrélées en composantes principales.  
**Résultat d'origine** : 0.95840

> ⚠️ Même remarque d'index que pour 8.2 : les composantes PCA ont un `RangeIndex` alors que `train_encoded` est indexé par `id`.

In [ ]:
X_best = train_encoded.join(service_cluster)   # baseline + Service_cluster

delay_pca = make_pca_components(train_encoded, delay_features)
delay_pca.head()

In [ ]:
X_delay_pca = X_best.join(delay_pca)
score_simple_split(X_delay_pca)

### 8.6 PCA sur les variables de confort
**Hypothèse** : `Inflight entertainment`, `Seat comfort` et `Cleanliness` sont fortement corrélées (cf. heatmap) ; la PCA peut condenser cette redondance.  
**Résultat d'origine** : 0.95826

In [ ]:
comfort_features = [
    "Inflight entertainment",
    "Seat comfort",
    "Cleanliness",
]
comfort_pca = make_pca_components(train_encoded, comfort_features)
comfort_pca.head()

In [ ]:
X_comfort_pca = X_best.join(comfort_pca)
score_simple_split(X_comfort_pca)

## 9. Round 2 – Split stratifié + target encoding
Nouveau protocole de validation (`score_stratified_split`) : split stratifié, et les features dépendant de la cible sont apprises **uniquement sur le train** (encodage out-of-fold pour le train).

**Hypothèse** : `Flight Distance` a 3 474 valeurs distinctes (cf. `nunique`) et une forte information mutuelle avec la cible ; un target encoding lissé capture mieux cette relation non linéaire que la valeur brute.

In [ ]:
baseline_score_stratified = score_stratified_split(train_encoded)
print(f"Score de base : {baseline_score_stratified:.5f}")      # origine : 0.95837

In [ ]:
flight_distance_encoding = make_target_encoding("Flight Distance", "sat_flight_dis")
score_stratified_split(train_encoded, feature_fn=flight_distance_encoding)   # origine : 0.95976

## 10. Modèle final & soumission
Features retenues : variables d'origine + `Service_cluster` + `sat_flight_dis` (target encoding de `Flight Distance`). Le modèle est entraîné sur tout le train, puis les probabilités sont prédites sur le test.

> ⚠️ Le clustering du test est ajusté sur le test lui-même (comme dans le code d'origine), les identifiants de clusters ne sont donc pas garantis identiques à ceux du train.

In [ ]:
test_df = pd.read_csv(TEST_PATH, index_col=ID_COLUMN)
test_encoded = encode_categoricals(test_df)
test_encoded.head()

In [ ]:
test_service_cluster = make_cluster_label(
    test_encoded, service_features, n_clusters=4, column_name="Service_cluster"
)
test_encoded = test_encoded.join(test_service_cluster)
test_encoded.head()

In [ ]:
train_final, test_final = flight_distance_encoding(train_encoded.copy(), test_encoded.copy())
train_final = train_final.join(service_cluster)

feature_columns = [col for col in train_encoded.columns if col != TARGET]
train_final = train_final[feature_columns + ["Service_cluster", "sat_flight_dis", TARGET]]
train_final.head()

In [ ]:
y_train_final = train_final.pop(TARGET)
final_model = XGBClassifier(random_state=SEED_MODEL, tree_method="hist").fit(train_final, y_train_final)

In [ ]:
test_proba = final_model.predict_proba(test_final)[:, 1]
test_proba

In [ ]:
submission = pd.DataFrame({ID_COLUMN: test_df.index, TARGET: test_proba})
submission.to_csv(SUBMISSION_PATH, index=False)
print(submission.head())
print("Submission shape:", submission.shape)